# 02 -- pdfplumber

## Objectifs pedagogiques

A la fin de ce notebook, vous saurez :

- extraire du texte avec positionnement precis ;
- extraire des tableaux et les convertir en DataFrames ;
- acceder aux caracteres, mots, lignes de facon structuree ;
- detecter les boites de texte et objets visuels ;
- filtrer par bounding box ;
- extraire les images ;
- construire un scraper PDF fiable.

## Prerequis -- ce que vous devez maitriser

- pypdf ou un autre outil de lecture PDF ;
- pandas pour stocker les tableaux extraits.

Ce que ce notebook **ne couvrira pas** :

- l'ecriture de PDF (voir `04_Reportlab_platypus.ipynb`) ;
- OCR sur PDF scannes (voir `03_Pymupdf.ipynb`) ;
- le layout analysis avance.

## Plan

1. Installation
2. Ouvrir un PDF
3. Extraction de texte
4. Caracteres, mots et lignes
5. Tables
6. Table vers DataFrame
7. Bounding box et filtres
8. Objets visuels
9. Images
10. Pipeline d'extraction
11. Synthese
12. Exercices

---

## 1. Installation

In [ ]:
# pip install pdfplumber
import pdfplumber
pdfplumber.__version__ if hasattr(pdfplumber, '__version__') else 'installed'

---

## 2. Ouvrir un PDF

In [ ]:
# Creer un PDF de test avec reportlab
import tempfile, pathlib
from reportlab.pdfgen import canvas
tmp = pathlib.Path(tempfile.gettempdir())
path = tmp / 'sample.pdf'
c2 = canvas.Canvas(str(path))
c2.drawString(72, 750, 'Titre du document')
c2.drawString(72, 720, 'Ligne 1 : contenu important')
c2.drawString(72, 700, 'Ligne 2 : autre information')
c2.save()
print('PDF demo cree')

In [ ]:
with pdfplumber.open(path) as pdf:
    print('Nb pages :', len(pdf.pages))
    page = pdf.pages[0]
    print('Dimensions :', page.width, 'x', page.height)

---

## 3. Extraction de texte

In [ ]:
with pdfplumber.open(path) as pdf:
    text = pdf.pages[0].extract_text()
    print(text)

---

## 4. Caracteres, mots et lignes

In [ ]:
with pdfplumber.open(path) as pdf:
    page = pdf.pages[0]
    print('chars :', len(page.chars))
    print('words :', len(page.extract_words()))
    print('lines :', len(page.lines))

In [ ]:
with pdfplumber.open(path) as pdf:
    words = pdf.pages[0].extract_words()
    for w in words[:3]:
        print(w['text'], '@', (w['x0'], w['top']))

---

## 5. Tables

In [ ]:
# Creer un PDF avec un tableau simple
from reportlab.platypus import SimpleDocTemplate, Table

doc_path = tmp / 'table.pdf'
doc = SimpleDocTemplate(str(doc_path))
table = Table([
    ['Nom', 'Age', 'Ville'],
    ['Alice', '30', 'Paris'],
    ['Bob', '25', 'Lyon'],
    ['Eve', '35', 'Marseille'],
])
table.setStyle([('GRID', (0, 0), (-1, -1), 0.5, (0, 0, 0))])
doc.build([table])
print('PDF table cree')

In [ ]:
with pdfplumber.open(doc_path) as pdf:
    page = pdf.pages[0]
    tables = page.extract_tables()
    print('Tables trouvees :', len(tables))
    for t in tables:
        for row in t:
            print(row)

---

## 6. Table vers DataFrame

In [ ]:
import pandas as pd
with pdfplumber.open(doc_path) as pdf:
    rows = pdf.pages[0].extract_table()
df = pd.DataFrame(rows[1:], columns=rows[0])
df

---

## 7. Bounding box et filtres

In [ ]:
with pdfplumber.open(path) as pdf:
    page = pdf.pages[0]
    # Zone haute (1/3 superieur)
    top_third = page.crop((0, 0, page.width, page.height / 3))
    print('Haut :', top_third.extract_text())

In [ ]:
# Filtrer des mots par position
with pdfplumber.open(path) as pdf:
    words = pdf.pages[0].extract_words()
    left_column = [w for w in words if w['x0'] < 300]
    print([w['text'] for w in left_column])

---

## 8. Objets visuels

In [ ]:
with pdfplumber.open(path) as pdf:
    page = pdf.pages[0]
    print('rects :', len(page.rects))
    print('curves :', len(page.curves))
    print('images :', len(page.images))

---

## 9. Images

In [ ]:
# with pdfplumber.open('doc_avec_images.pdf') as pdf:
#     for i, img in enumerate(pdf.pages[0].images):
#         print(img['name'], img['x0'], img['top'])
print('Images accessibles via page.images')

---

## 10. Pipeline d'extraction

In [ ]:
def extract_all_tables(path: str) -> list[pd.DataFrame]:
    out = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            for table in page.extract_tables():
                if table and table[0]:
                    out.append(pd.DataFrame(table[1:], columns=table[0]))
    return out

tables = extract_all_tables(str(doc_path))
len(tables)

---

## Synthese

| Operation | pdfplumber |
|---|---|
| Ouvrir | `pdfplumber.open(path)` |
| Pages | `pdf.pages` |
| Texte | `page.extract_text()` |
| Mots | `page.extract_words()` |
| Tables | `page.extract_tables()` |
| Crop | `page.crop((x0, top, x1, bottom))` |
| Dimensions | `page.width`, `page.height` |
| Objets | `page.chars`, `page.rects`, `page.curves`, `page.images` |

### Regles a retenir

- **pdfplumber = meilleure extraction texte** que pypdf pour les PDF structures.
- **`extract_tables`** detecte les tableaux par lignes et alignements, pas toujours parfait.
- **Crop + extract** pour filtrer une zone.
- **Coordonnees PDF** : origin en bas-gauche normalement, pdfplumber utilise le haut-gauche.

---

## Exercices

### Exercice 1 (facile) -- Extraire texte d'une page

Fonction `page_text(path, page_num)` qui retourne le texte d'une page donnee.

In [ ]:
import pdfplumber

# TODO

<details>
<summary>Voir la correction</summary>

```python
import pdfplumber

def page_text(path: str, page_num: int) -> str:
    with pdfplumber.open(path) as pdf:
        return pdf.pages[page_num].extract_text() or ''
```

</details>

### Exercice 2 (facile) -- Nombre de mots

Fonction `word_count(path)` qui compte le total de mots dans tout le PDF.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import pdfplumber

def word_count(path: str) -> int:
    total = 0
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            total += len(page.extract_words())
    return total
```

</details>

### Exercice 3 (moyen) -- Table vers CSV

Fonction `first_table_to_csv(pdf_path, csv_path)` qui extrait la premiere table et l'ecrit en CSV.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import pdfplumber
import csv

def first_table_to_csv(pdf_path: str, csv_path: str) -> None:
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            table = page.extract_table()
            if table:
                with open(csv_path, 'w', newline='') as f:
                    csv.writer(f).writerows(table)
                return
```

</details>

### Exercice 4 (moyen) -- Extraire entete

Fonction `extract_header(path)` qui retourne le texte present dans le 1/4 superieur de la premiere page.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import pdfplumber

def extract_header(path: str) -> str:
    with pdfplumber.open(path) as pdf:
        page = pdf.pages[0]
        header = page.crop((0, 0, page.width, page.height / 4))
        return header.extract_text() or ''
```

</details>

### Exercice 5 (difficile) -- Detection de colonnes

Pour une page avec 2 colonnes, ecrivez une fonction qui extrait les deux colonnes separement en filtrant par x0.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import pdfplumber

def two_columns(path: str) -> tuple[str, str]:
    with pdfplumber.open(path) as pdf:
        page = pdf.pages[0]
        mid = page.width / 2
        left = page.crop((0, 0, mid, page.height)).extract_text() or ''
        right = page.crop((mid, 0, page.width, page.height)).extract_text() or ''
        return left, right
```

</details>

---

## Ressources externes

- [pdfplumber GitHub](https://github.com/jsvine/pdfplumber)
- [pdfplumber README](https://github.com/jsvine/pdfplumber#readme)